# MV-STGNN ablations
## Which parts actually earn their place?

**Context:** Companion to pooled_mvstgnn_5fold and baselines_pooled using the same pooled protocol, normalization, folds, and significance machinery (comparison family F2).

**Reference Model (A0):** Retrained within this study to maintain a consistent epoch budget across all ablations and serve as a reproducibility check against the 70-epoch reference (bal_acc 0.8009 ± 0.0574).

**Training Budget:** Epoch cap increased to $90$ (patience $12$) to prevent the cap-binding seen in 3 of 5 folds during earlier 70-epoch runs; best_epoch and epochs_run are tracked per variant to expose any remaining saturation.

**Two-Stage Screening:** Optional FOLDS_TO_RUN=(0,) screens all variants cheaply on Fold 0 (the hardest fold with test rep 1) before promoting survivors to all 5 folds for paired testing.

---
## Step 0 — Environment

In [ ]:
import hashlib, json, math, os, random, sys, time, warnings
from dataclasses import dataclass, asdict, replace as dc_replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, f1_score)

warnings.filterwarnings("ignore", category=RuntimeWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
BF16 = torch.cuda.is_bf16_supported() if DEVICE.type == "cuda" else False
print("torch", torch.__version__, "| device", DEVICE, "| bf16", BF16)

torch 2.10.0+cu128 | device cuda | bf16 True


---
## Step 1 — Config with ablation switches

Every switch below is a genuine architectural or training knob, defaulting to the reference
configuration. An ablation is expressed purely as a dict of overrides, so there is one model
implementation and no risk of the "ablated" code drifting from the real one.

In [ ]:
PROJECT = Path(r"c:\Users\deskt\Desktop\Nianpro_EMG_Project")
DB2_ROOT = PROJECT / "nina_pro_db_2" / "DB2_Extracted"
RESULTS = PROJECT / "results" / "tables"
RUNS = PROJECT / "results" / "runs" / "ablations"
PREDS = PROJECT / "results" / "preds_abl"
for d in (RESULTS, RUNS, PREDS):
    d.mkdir(parents=True, exist_ok=True)

GNN_TAG = "mvstgnn_pooled_5cls_200ms"


def _rel(p):
    try:
        return str(Path(p).relative_to(PROJECT))
    except ValueError:
        return str(p)


ALL_VIEWS = ("anat", "coh", "dyn", "learn", "uniform")


@dataclass
class Cfg:
    # ======== DATA / PROTOCOL (must match the reference run) ==============
    fs: int = 2000
    n_channels: int = 12
    include_rest: bool = False
    subjects: tuple = tuple(range(1, 21))
    class_subset: tuple = (0, 1, 2, 3, 4)
    bp_low: float = 20.0
    bp_high: float = 450.0
    bp_order: int = 4
    notch_freqs: tuple = (50, 100, 150, 200, 250, 300, 350, 400)
    notch_q: float = 30.0
    trim_ms: int = 50
    win_ms: int = 200                 # ABLATED by A12*
    train_stride_ms: int = 40
    eval_stride_ms: int = 50
    norm_pct: float = 99.9
    ring_channels: tuple = (0, 1, 2, 3, 4, 5, 6, 7)
    floor_frac: float = 0.05
    bad_channels: tuple = ((7, 5), (20, 5))
    val_rep: int = 3
    test_reps: tuple = (1, 2, 4, 5, 6)

    # ======== GRAPH (ablation surface) ====================================
    use_graph: bool = True            # A1: False -> nodes never exchange information
    graph_views: tuple = ("anat", "coh", "dyn", "learn")   # A2-A6, A1b
    coh_global: bool = False          # P2: population-mean A_coh instead of per-subject
    graph_topk: int = 6               # A11: 12 -> dense A_dyn
    heads_per_view: int = 2
    d_attn: int = 32
    emb_dim: int = 16
    coh_band: tuple = (20.0, 150.0)
    coh_nperseg: int = 256
    coh_max_frames: int = 4000

    # ======== MODEL (ablation surface) ====================================
    ms_kernels: tuple = (3, 9, 27)
    stem_width: int = 16
    d_model: int = 64                 # A15*
    n_st_blocks: int = 3
    use_synergy: bool = True          # A7
    n_synergies: int = 4              # A8*
    head_hidden: int = 256
    p_drop: float = 0.2
    p_drop_head: float = 0.3
    stem_norm: str = "group"          # P3: "batch"
    norm_groups: int = 8
    use_subject_film: bool = True     # P1
    subj_emb_dim: int = 16

    # ======== LOSS (ablation surface) =====================================
    label_smoothing: float = 0.05
    w_supcon: float = 0.10            # A9: 0.0
    supcon_tau: float = 0.10
    w_linkpred: float = 0.01
    w_entropy: float = 0.01
    w_adj_l1: float = 1e-4
    class_weighted: bool = True
    cap_train_per_class: bool = False

    # ======== OPTIMISATION (fixed across all variants) ====================
    lr: float = 1.5e-3
    weight_decay: float = 1e-2
    warmup_epochs: int = 5
    epochs: int = 90
    batch_size: int = 256
    eval_batch: int = 512
    patience: int = 12
    grad_clip: float = 1.0
    amp: bool = True

    # ======== AUGMENTATION (ablation surface) =============================
    aug_noise_snr_db: tuple = (20.0, 35.0)
    aug_amp_scale: tuple = (0.9, 1.1)
    aug_chan_drop_p: float = 0.15     # A10: 0.0
    aug_time_mask_ms: int = 20

    @property
    def win(self): return int(round(self.win_ms * self.fs / 1000))
    @property
    def trim(self): return int(round(self.trim_ms * self.fs / 1000))
    @property
    def train_stride(self): return int(round(self.train_stride_ms * self.fs / 1000))
    @property
    def eval_stride(self): return int(round(self.eval_stride_ms * self.fs / 1000))
    @property
    def n_classes(self):
        if self.class_subset is not None:
            return len(self.class_subset)
        return 18 if self.include_rest else 17
    @property
    def label_map(self):
        if self.class_subset is None:
            return None
        return {c: i for i, c in enumerate(self.class_subset)}
    @property
    def n_movement_classes(self):
        return len(self.class_subset) if self.class_subset is not None else 17
    @property
    def n_folds(self): return len(self.test_reps)

    def train_reps(self, test_rep):
        return tuple(r for r in range(1, 7) if r not in (self.val_rep, test_rep))

    def bad_for(self, subject):
        return [c for s, c in self.bad_channels if s == subject]


BASE = Cfg()
for v in BASE.graph_views:
    assert v in ALL_VIEWS, f"unknown view {v}"

DATA_FIELDS = ("fs", "n_channels", "include_rest", "subjects", "class_subset",
               "bp_low", "bp_high", "bp_order", "notch_freqs", "notch_q",
               "trim_ms", "win_ms", "train_stride_ms", "eval_stride_ms",
               "norm_pct", "ring_channels", "floor_frac", "bad_channels",
               "val_rep", "test_reps")


def canon(v):
    return json.loads(json.dumps(v))


def cfg_hash(cfg):
    return hashlib.md5(json.dumps(canon(asdict(cfg)), sort_keys=True).encode()).hexdigest()[:8]


def latest(pattern):
    hits = sorted(RESULTS.glob(pattern))
    if not hits:
        return None
    named = [p for p in hits if "latest" in p.name]
    return named[0] if named else hits[-1]


ref_cfg_path = latest(f"{GNN_TAG}_config_*.json")
if ref_cfg_path is not None:
    REF = json.loads(ref_cfg_path.read_text())
    mine = canon(asdict(BASE))
    diff = [k for k in DATA_FIELDS if mine.get(k) != canon(REF.get(k))]
    print(f"reference config: {ref_cfg_path.name}")
    if diff:
        print(f"  [WARN] data fields differ from the reference run: {diff}")
        print("  -> ablations remain internally paired, but are NOT paired with that run")
    else:
        print("  [PASS] all data/protocol fields match the reference MV-STGNN run")
    print(f"  epochs: ablations {BASE.epochs} vs reference {REF.get('epochs')}"
          f"  ({'same' if BASE.epochs == REF.get('epochs') else 'A0 is retrained here'})")
else:
    print("[WARN] no reference config found; A0 will still be trained here")

print(f"\nBASE hash {cfg_hash(BASE)} | {len(BASE.subjects)} subjects x "
      f"{BASE.n_classes} classes | chance {1.0/BASE.n_classes:.4f}")

reference config: mvstgnn_pooled_5cls_200ms_config_20260806_233411.json
  [PASS] all data/protocol fields match the reference MV-STGNN run
  epochs: ablations 90 vs reference 70  (A0 is retrained here)

BASE hash d2429ed3 | 20 subjects x 5 classes | chance 0.2000


---
## Step 2 — Shared data pipeline

Identical to notebooks 02 and 03. Reproduced rather than imported so the notebook is
self-contained; Step 4 verifies the resulting splits with the same fingerprint used there.

In [ ]:
def build_filter_cascade(cfg):
    sec = [signal.butter(cfg.bp_order, [cfg.bp_low, cfg.bp_high],
                         btype="band", fs=cfg.fs, output="sos")]
    for f0 in cfg.notch_freqs:
        if f0 < cfg.fs / 2:
            b, a = signal.iirnotch(f0, cfg.notch_q, fs=cfg.fs)
            sec.append(signal.tf2sos(b, a))
    return np.concatenate(sec, axis=0)


SOS = build_filter_cascade(BASE)


def load_raw(subject, cfg):
    path = DB2_ROOT / f"DB2_s{subject}" / f"S{subject}_E1_A1.mat"
    try:
        m = scipy.io.loadmat(str(path))
        emg = np.asarray(m["emg"], dtype=np.float32)
        rs = np.asarray(m["restimulus"]).ravel().astype(np.int8)
        rr = np.asarray(m["rerepetition"]).ravel().astype(np.int8)
    except NotImplementedError:
        import h5py
        with h5py.File(path, "r") as f:
            emg = np.asarray(f["emg"], dtype=np.float32).T
            rs = np.asarray(f["restimulus"]).ravel().astype(np.int8)
            rr = np.asarray(f["rerepetition"]).ravel().astype(np.int8)
    assert emg.shape[1] == cfg.n_channels and len(rs) == len(rr) == len(emg)
    return emg, rs, rr


def filter_signal(emg, sos):
    return np.ascontiguousarray(
        signal.sosfiltfilt(sos, emg.astype(np.float64), axis=0), dtype=np.float32)


def extract_segments(rs, rr, cfg):
    key = rs.astype(np.int64) * 100 + rr.astype(np.int64)
    brk = np.flatnonzero(np.diff(key)) + 1
    starts, ends = np.r_[0, brk], np.r_[brk, len(key)]
    raw = [(int(s), int(e), int(rs[s]), int(rr[s])) for s, e in zip(starts, ends)]
    if cfg.include_rest:
        raw = [(s, e, lab, (next((raw[j][3] for j in range(i + 1, len(raw))
                                 if raw[j][2] != 0), 0) if lab == 0 else rep))
               for i, (s, e, lab, rep) in enumerate(raw)]
    lmap = cfg.label_map
    segs, dropped = [], 0
    for s, e, lab, rep in raw:
        if lab == 0 and not cfg.include_rest:
            continue
        if not 1 <= rep <= 6:
            dropped += 1
            continue
        y = 17 if lab == 0 else lab - 1
        if lmap is not None:
            if y not in lmap:
                continue
            y = lmap[y]
        s2, e2 = s + cfg.trim, e - cfg.trim
        if e2 - s2 < cfg.win:
            dropped += 1
            continue
        segs.append((s2, e2, y, rep))
    return segs, dropped


def build_windows(segs, stride, cfg):
    st_, lb_, rp_, sg_ = [], [], [], []
    for sid, (s, e, y, rep) in enumerate(segs):
        last = e - cfg.win
        if last < s:
            continue
        st = np.arange(s, last + 1, stride, dtype=np.int64)
        st_.append(st)
        lb_.append(np.full(len(st), y, dtype=np.int64))
        rp_.append(np.full(len(st), rep, dtype=np.int64))
        sg_.append(np.full(len(st), sid, dtype=np.int64))
    if not st_:
        return {k: np.zeros(0, np.int64) for k in ("start", "label", "rep", "seg_id")}
    return dict(start=np.concatenate(st_), label=np.concatenate(lb_),
                rep=np.concatenate(rp_), seg_id=np.concatenate(sg_))


def subset(win, mask):
    return {k: v[mask] for k, v in win.items()}


KEYS = ("start", "label", "rep", "seg_id", "subj_idx")


def build_pooled_index(SUBJ, subjects, offsets, stride, reps, cfg):
    parts = []
    for si, s in enumerate(subjects):
        w = build_windows(SUBJ[s]["segs"], stride, cfg)
        w = subset(w, np.isin(w["rep"], reps))
        w["start"] = w["start"] + offsets[si]
        w["seg_id"] = w["seg_id"] + si * 1000
        w["subj_idx"] = np.full(len(w["start"]), si, dtype=np.int64)
        parts.append(w)
    return {k: np.concatenate([p[k] for p in parts]) for k in KEYS}


def fold_indices(SUBJ, subjects, offsets, fold, cfg):
    test_rep = cfg.test_reps[fold]
    train_reps = cfg.train_reps(test_rep)
    tr = build_pooled_index(SUBJ, subjects, offsets, cfg.train_stride, train_reps, cfg)
    va = build_pooled_index(SUBJ, subjects, offsets, cfg.eval_stride, (cfg.val_rep,), cfg)
    te = build_pooled_index(SUBJ, subjects, offsets, cfg.eval_stride, (test_rep,), cfg)
    return tr, va, te, dict(fold=fold, test_rep=test_rep, val_rep=cfg.val_rep,
                            train_reps=list(train_reps), n_train=len(tr["start"]),
                            n_val=len(va["start"]), n_test=len(te["start"]))


def compute_scale(sig, segs, train_reps, cfg):
    mask = np.zeros(len(sig), dtype=bool)
    for s, e, _, rep in segs:
        if rep in train_reps:
            mask[s:e] = True
    p = np.percentile(np.abs(sig[mask]), cfg.norm_pct, axis=0).astype(np.float64)
    ref = float(np.median(p[list(cfg.ring_channels)]))
    return np.maximum(p, cfg.floor_frac * ref).astype(np.float32), \
        int((p < cfg.floor_frac * ref).sum())


class PooledSource:
    def __init__(self, n_total, cfg, device=DEVICE):
        self.cfg, self.device = cfg, device
        self.buf = torch.zeros((n_total, cfg.n_channels), dtype=torch.float16, device=device)
        self.ar = torch.arange(cfg.win, device=device)

    def write(self, offset, sig, scale):
        x = np.clip(sig / scale, -1.0, 1.0)
        self.buf[offset:offset + len(x)] = torch.from_numpy(x).to(self.device, torch.float16)

    def gather(self, starts, train, gen=None):
        idx = starts[:, None] + self.ar[None, :]
        x = self.buf[idx].to(torch.float32).permute(0, 2, 1).contiguous()
        return self._augment(x, gen) if train else x

    def _augment(self, x, gen=None):
        cfg, dev = self.cfg, x.device
        B, Ch, L = x.shape
        r = lambda *s: torch.rand(*s, device=dev, generator=gen)
        lo, hi = cfg.aug_amp_scale
        x = x * (lo + (hi - lo) * r(B, Ch, 1))
        slo, shi = cfg.aug_noise_snr_db
        snr = slo + (shi - slo) * r(B, 1, 1)
        rms = x.pow(2).mean(dim=2, keepdim=True).sqrt()
        x = x + rms * torch.pow(10.0, -snr / 20.0) * torch.randn(
            x.shape, device=dev, generator=gen)
        if cfg.aug_chan_drop_p > 0:
            hit = (r(B, 1) < cfg.aug_chan_drop_p).to(x.dtype)
            which = torch.randint(0, Ch, (B,), device=dev, generator=gen)
            x = x * (1.0 - F.one_hot(which, Ch).to(x.dtype) * hit).unsqueeze(-1)
        span = int(round(cfg.aug_time_mask_ms * cfg.fs / 1000))
        if span > 0:
            t0 = torch.randint(0, max(1, L - span), (B, 1), device=dev, generator=gen)
            ta = torch.arange(L, device=dev)[None, :]
            x = x * (~((ta >= t0) & (ta < t0 + span)))[:, None, :].to(x.dtype)
        return x


class Batcher:
    def __init__(self, win, device=DEVICE):
        self.start = torch.from_numpy(win["start"]).to(device)
        self.label = torch.from_numpy(win["label"]).to(device)
        self.subj = torch.from_numpy(win["subj_idx"]).to(device)
        self.n = len(win["start"])
        self.device = device

    def epoch(self, batch, shuffle, rng=None):
        order = np.arange(self.n)
        if shuffle:
            (rng or np.random).shuffle(order)
        for i in range(0, self.n, batch):
            sel = torch.from_numpy(order[i:i + batch]).to(self.device)
            yield self.start[sel], self.label[sel], self.subj[sel]


def build_A_anat(n=12):
    A = np.zeros((n, n), dtype=np.float64)

    def link(i, j, w):
        A[i, j] = max(A[i, j], w); A[j, i] = max(A[j, i], w)

    for i in range(8):
        link(i, (i + 1) % 8, 1.0)
        link(i, (i + 2) % 8, 0.5)
    for c in (8, 9):
        for i in range(8):
            link(c, i, 0.3)
    link(8, 9, 0.3)
    link(10, 11, 1.0)
    for c in (10, 11):
        for i in range(8):
            link(c, i, 0.1)
    A = A + np.eye(n)
    dm = 1.0 / np.sqrt(A.sum(1))
    return (A * dm[:, None] * dm[None, :]).astype(np.float32)


def sym_norm_topk(A, topk):
    n = A.shape[0]
    M = np.array(A, dtype=np.float64)
    np.fill_diagonal(M, 0.0)
    keep = np.zeros_like(M, dtype=bool)
    if topk < n - 1:
        for i in range(n):
            keep[i, np.argsort(-M[i])[:topk]] = True
        keep |= keep.T
    else:
        keep[:] = True
    M = M * keep + np.eye(n)
    dm = 1.0 / np.sqrt(np.maximum(M.sum(1), 1e-12))
    return (M * dm[:, None] * dm[None, :]).astype(np.float32)


def compute_A_coh(sig, segs, train_reps, cfg, seed=0):
    nper, hop = cfg.coh_nperseg, cfg.coh_nperseg // 2
    frames = []
    for s, e, _, rep in segs:
        if rep not in train_reps:
            continue
        seg = sig[s:e]
        nf = (len(seg) - nper) // hop + 1
        if nf > 0:
            idx = np.arange(nper)[None, :] + (np.arange(nf) * hop)[:, None]
            frames.append(seg[idx])
    if not frames:
        return build_A_anat(cfg.n_channels)
    Fr = np.concatenate(frames, axis=0)
    if len(Fr) > cfg.coh_max_frames:
        Fr = Fr[np.random.default_rng(seed).choice(len(Fr), cfg.coh_max_frames, replace=False)]
    X = np.fft.rfft(Fr * np.hanning(nper).astype(np.float32)[None, :, None], axis=1)
    fr = np.fft.rfftfreq(nper, 1.0 / cfg.fs)
    X = X[:, (fr >= cfg.coh_band[0]) & (fr <= cfg.coh_band[1]), :]
    nf = X.shape[0]
    Pxy = np.einsum("nfi,nfj->fij", X, np.conj(X)) / nf
    Pxx = np.real(np.einsum("nfi,nfi->fi", X, np.conj(X))) / nf
    msc = np.abs(Pxy) ** 2 / (Pxx[:, :, None] * Pxx[:, None, :] + 1e-20)
    return sym_norm_topk(msc.mean(axis=0), cfg.graph_topk)


def metrics_from(y, p, seg, n_classes):
    out = dict(acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
               macro_f1=f1_score(y, p, average="macro", zero_division=0),
               weighted_f1=f1_score(y, p, average="weighted", zero_division=0),
               kappa=cohen_kappa_score(y, p))
    sy, sp = [], []
    for s in np.unique(seg):
        m = seg == s
        sy.append(y[m][0]); sp.append(np.bincount(p[m], minlength=n_classes).argmax())
    out["vote_acc"] = accuracy_score(sy, sp)
    return out


def per_subject_rows(win, pred, subjects, n_classes):
    rows = []
    for si, s in enumerate(subjects):
        m = win["subj_idx"] == si
        if not m.any():
            continue
        r = metrics_from(win["label"][m], pred[m], win["seg_id"][m], n_classes)
        r["subject"] = s
        r["n_test"] = int(m.sum())
        rows.append(r)
    return rows


def make_class_weights(labels, n_classes, device=DEVICE):
    cnt = np.maximum(np.bincount(labels, minlength=n_classes).astype(np.float64), 1.0)
    return torch.tensor(cnt.sum() / (n_classes * cnt), dtype=torch.float32, device=device)


def cap_train_windows(win, n_classes, seed=0):
    rng = np.random.default_rng(seed)
    cnt = np.bincount(win["label"], minlength=n_classes)
    present = cnt[cnt > 0]
    if not len(present):
        return win
    target = int(present.min())
    keep = [rng.choice(i, target, replace=False) if len(i) > target else i
            for i in (np.flatnonzero(win["label"] == c) for c in range(n_classes)) if len(i)]
    return subset(win, np.sort(np.concatenate(keep)))


def supcon_loss(z, y, tau):
    z = F.normalize(z.float(), dim=1)
    sim = z @ z.t() / tau
    n = z.shape[0]
    eye = torch.eye(n, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, torch.finfo(sim.dtype).min)
    pos = (y[:, None] == y[None, :]) & ~eye
    npos = pos.sum(1)
    valid = npos > 0
    if not valid.any():
        return z.new_zeros(())
    lp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    return -(torch.where(pos, lp, torch.zeros_like(lp)).sum(1)[valid] / npos[valid]).mean()


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


A_ANAT_NP = build_A_anat(BASE.n_channels)
A_ANAT = torch.from_numpy(A_ANAT_NP).to(DEVICE)
print("pipeline defined")

pipeline defined


---
## Step 3 — Load subjects and gate the splits

In [ ]:
def load_all_subjects(subjects, cfg, verbose=True):
    SUBJ, t0 = {}, time.time()
    for i, s in enumerate(subjects):
        emg, rs, rr = load_raw(s, cfg)
        for c in cfg.bad_for(s):
            emg[:, c] = 0.0
        sig = filter_signal(emg, SOS)
        segs, _ = extract_segments(rs, rr, cfg)
        if not cfg.include_rest:
            exp = 6 * cfg.n_movement_classes
            assert len(segs) == exp, f"subject {s}: {len(segs)} segs, expected {exp}"
        SUBJ[s] = dict(sig=sig, segs=segs, n=len(sig), bad=cfg.bad_for(s))
    lens = np.array([SUBJ[s]["n"] for s in subjects], dtype=np.int64)
    if verbose:
        print(f"loaded {len(subjects)} subjects in {time.time()-t0:.0f}s, "
              f"{lens.sum():,} samples, RAM {lens.sum()*12*4/1e9:.2f} GB")
    return SUBJ, np.r_[0, np.cumsum(lens)[:-1]], int(lens.sum())


SUBJECTS = list(BASE.subjects)
SUBJ, OFFSETS, N_TOTAL = load_all_subjects(SUBJECTS, BASE)


def split_fingerprint(SUBJ, subjects, offsets, cfg):
    h = hashlib.md5()
    for f in range(cfg.n_folds):
        for w in fold_indices(SUBJ, subjects, offsets, f, cfg)[:3]:
            for k in ("start", "label", "rep", "subj_idx"):
                h.update(np.ascontiguousarray(w[k]).tobytes())
    return h.hexdigest()[:16]


def leakage_gate(SUBJ, subjects, offsets, n_total, cfg, verbose=False):
    lens = np.array([SUBJ[s]["n"] for s in subjects])
    ends = offsets + lens
    assert np.all(offsets[1:] == ends[:-1]) and int(ends[-1]) == n_total
    worst_all = 0
    for f in range(cfg.n_folds):
        tr, va, te, m = fold_indices(SUBJ, subjects, offsets, f, cfg)
        for w in (tr, va, te):
            lo = offsets[w["subj_idx"]]
            assert np.all(w["start"] >= lo)
            assert np.all(w["start"] + cfg.win <= lo + lens[w["subj_idx"]])
        assert set(te["rep"]) == {m["test_rep"]} and set(va["rep"]) == {cfg.val_rep}
        assert set(tr["rep"]) == set(m["train_reps"])
        assert all(len(np.unique(w["subj_idx"])) == len(subjects) for w in (tr, va, te))
        for si in range(len(subjects)):
            cov = {}
            for nm, w in (("tr", tr), ("va", va), ("te", te)):
                ws = w["start"][w["subj_idx"] == si] - offsets[si]
                c = np.zeros(lens[si], dtype=bool)
                if len(ws):
                    c[(ws[:, None] + np.arange(cfg.win)[None, :]).ravel()] = True
                cov[nm] = c
            worst_all = max(worst_all, int((cov["tr"] & cov["va"]).sum()),
                            int((cov["tr"] & cov["te"]).sum()),
                            int((cov["va"] & cov["te"]).sum()))
    assert worst_all == 0, f"sample overlap between splits: {worst_all}"
    assert cfg.val_rep not in cfg.test_reps
    return True


leakage_gate(SUBJ, SUBJECTS, OFFSETS, N_TOTAL, BASE)
BASE_FP = split_fingerprint(SUBJ, SUBJECTS, OFFSETS, BASE)
print(f"leakage gate GREEN | base split fingerprint {BASE_FP}")
print("  (window-length ablations A12* change the windows, so they carry their own "
      "fingerprint — flagged as data_changed in the results)")

loaded 20 subjects in 41s, 36,027,660 samples, RAM 1.73 GB
leakage gate GREEN | base split fingerprint 3fed90994b841e0d
  (window-length ablations A12* change the windows, so they carry their own fingerprint — flagged as data_changed in the results)


---
## Step 4 — Configurable MV-STGNN

**One implementation, driven by flags.** The ablated model is the real model with a switch
flipped, so there is no chance of the "ablated" branch drifting from the version being claimed.

`use_graph=False` (**A1**) replaces multi-view graph attention with a per-node `Linear`, so nodes
never exchange information anywhere in the ST blocks — electrodes only combine at the readout.
Together with **A4** (a free dense learned adjacency) this brackets the question cleanly:

```
A1  no node mixing at all
A1b uniform mixing (every electrode averaged equally)   ← simplest possible graph
A4  unstructured learned N×N mixing
A0  structured multi-view mixing                        ← the claim
```

In [ ]:
def make_norm(cfg, c):
    if cfg.stem_norm == "batch":
        return nn.BatchNorm1d(c)
    return nn.GroupNorm(math.gcd(cfg.norm_groups, c) or 1, c)


class TCNBlock(nn.Module):
    def __init__(self, cfg, cin, cout, k, dil, p):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout, k, dilation=dil, padding=dil * (k - 1) // 2)
        self.bn = make_norm(cfg, cout); self.act = nn.GELU(); self.do = nn.Dropout(p)
        self.res = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        return self.do(self.act(self.bn(self.conv(x)))) + self.res(x)


class SubjectFiLM(nn.Module):
    def __init__(self, n_subjects, emb_dim, d):
        super().__init__()
        self.emb = nn.Embedding(n_subjects, emb_dim)
        self.to_gb = nn.Linear(emb_dim, 2 * d)
        nn.init.normal_(self.emb.weight, std=0.02)
        nn.init.zeros_(self.to_gb.weight); nn.init.zeros_(self.to_gb.bias)

    def forward(self, h, subj):
        g, b = self.to_gb(self.emb(subj)).chunk(2, dim=-1)
        return h * (1.0 + g[:, None, None, :]) + b[:, None, None, :]


class Stem(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ms = nn.ModuleList([nn.Conv1d(1, cfg.stem_width, k, padding=k // 2)
                                 for k in cfg.ms_kernels])
        c = cfg.stem_width * len(cfg.ms_kernels)
        self.bn = make_norm(cfg, c); self.act = nn.GELU(); self.pool = nn.MaxPool1d(4)
        d = cfg.d_model
        self.b1 = TCNBlock(cfg, c, d, 5, 1, cfg.p_drop)
        self.b2 = TCNBlock(cfg, d, d, 5, 2, cfg.p_drop)
        self.b3 = TCNBlock(cfg, d, d, 5, 4, cfg.p_drop)
        self.dn = nn.AvgPool1d(2)

    def forward(self, x):
        B, Ch, L = x.shape
        h = x.reshape(B * Ch, 1, L)
        h = torch.cat([m(h) for m in self.ms], dim=1)
        h = self.pool(self.act(self.bn(h)))
        h = self.dn(self.b1(h)); h = self.dn(self.b2(h)); h = self.b3(h)
        D, T = h.shape[1], h.shape[2]
        return h.reshape(B, Ch, D, T).permute(0, 3, 1, 2)


def gconv(A, vv):
    A = A.to(vv.dtype)
    if A.dim() == 2:
        return torch.einsum("ij,bthjd->bthid", A, vv)
    if A.dim() == 3:
        return torch.einsum("bij,bthjd->bthid", A, vv)
    return torch.einsum("btij,bthjd->bthid", A, vv)


def as_pooled_adj(A, B, N, dtype):
    A = A.to(dtype)
    if A.dim() == 2:
        return A.expand(B, N, N)
    return A if A.dim() == 3 else A.mean(1)


class NodeMLP(nn.Module):
    """A1 control: same depth/params shape, but NO cross-node information flow."""

    def __init__(self, cfg):
        super().__init__()
        self.lin = nn.Linear(cfg.d_model, cfg.d_model)

    def forward(self, h, A_anat=None, A_coh=None):
        return self.lin(h), None, None


class MultiViewGraphConv(nn.Module):
    def __init__(self, cfg, n_nodes, views):
        super().__init__()
        self.views = tuple(views)
        self.hpv = cfg.heads_per_view
        self.H = max(1, len(self.views) * self.hpv)
        d = cfg.d_model
        self.hd = max(1, d // self.H)
        self.proj = nn.Linear(d, self.H * self.hd)
        self.out = nn.Linear(self.H * self.hd, d)
        self.needs_dyn = "dyn" in self.views
        self.needs_learn = "learn" in self.views
        if self.needs_dyn:
            self.q = nn.Linear(d, cfg.d_attn); self.k = nn.Linear(d, cfg.d_attn)
        if self.needs_learn:
            self.E1 = nn.Parameter(torch.randn(n_nodes, cfg.emb_dim) * 0.1)
            self.E2 = nn.Parameter(torch.randn(n_nodes, cfg.emb_dim) * 0.1)
        self.topk = min(cfg.graph_topk, n_nodes)
        self.register_buffer("A_uni", torch.full((n_nodes, n_nodes), 1.0 / n_nodes))

    def forward(self, h, A_anat=None, A_coh=None):
        B, T, N, D = h.shape
        v = self.proj(h).view(B, T, N, self.H, self.hd).permute(0, 1, 3, 2, 4)
        built, A_dyn = {}, None
        if self.needs_dyn:
            q, k = self.q(h), self.k(h)
            s = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(q.shape[-1])
            if self.topk < N:
                s = s.masked_fill(s < s.topk(self.topk, dim=-1).values[..., -1:],
                                  float("-inf"))
            A_dyn = s.softmax(-1)
            built["dyn"] = A_dyn
        if self.needs_learn:
            built["learn"] = torch.softmax(F.relu(self.E1 @ self.E2.t()), dim=-1)
        if "anat" in self.views:
            built["anat"] = A_anat
        if "coh" in self.views:
            built["coh"] = A_coh
        if "uniform" in self.views:
            built["uniform"] = self.A_uni

        parts, pooled = [], []
        for vi, name in enumerate(self.views):
            A = built[name]
            parts.append(gconv(A, v[:, :, vi * self.hpv:(vi + 1) * self.hpv]))
            pooled.append(as_pooled_adj(A, B, N, h.dtype))
        o = torch.cat(parts, dim=2).permute(0, 1, 3, 2, 4).reshape(B, T, N, self.H * self.hd)
        return self.out(o), torch.stack(pooled, 0).mean(0), A_dyn


class TemporalGate(nn.Module):
    def __init__(self, d, k=5, dil=1):
        super().__init__()
        self.conv = nn.Conv1d(d, 2 * d, k, dilation=dil,
                              padding=dil * (k - 1) // 2, groups=d)

    def forward(self, h):
        B, T, N, D = h.shape
        x = h.permute(0, 2, 3, 1).reshape(B * N, D, T)
        y = self.conv(x).view(B * N, D, 2, T)
        y = torch.tanh(y[:, :, 0]) * torch.sigmoid(y[:, :, 1])
        return y.view(B, N, D, T).permute(0, 3, 1, 2)


class STBlock(nn.Module):
    def __init__(self, cfg, n_nodes, dil, views):
        super().__init__()
        d = cfg.d_model
        self.g = MultiViewGraphConv(cfg, n_nodes, views) if (cfg.use_graph and views) \
            else NodeMLP(cfg)
        self.t = TemporalGate(d, 5, dil)
        self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.do = nn.Dropout(cfg.p_drop)

    def forward(self, h, A_anat=None, A_coh=None):
        g, A_f, A_d = self.g(self.n1(h), A_anat, A_coh)
        h = h + self.do(g)
        h = h + self.do(self.t(self.n2(h)))
        return h, A_f, A_d


class SynergyPool(nn.Module):
    def __init__(self, d, K):
        super().__init__()
        self.lin = nn.Linear(d, K)

    def forward(self, h):
        S = torch.softmax(self.lin(h.mean(1)), dim=-1)
        return torch.einsum("bnk,btnd->btkd", S.to(h.dtype), h), S


class AttnReadout(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.score = nn.Linear(d, 1)

    def forward(self, h):
        z = (torch.softmax(self.score(h), dim=2) * h).sum(2)
        return torch.cat([z.mean(1), z.max(1).values], dim=-1)


class MVSTGNN(nn.Module):
    def __init__(self, cfg, n_subjects=20):
        super().__init__()
        self.cfg = cfg
        N, K, d = cfg.n_channels, cfg.n_synergies, cfg.d_model
        views = tuple(cfg.graph_views) if cfg.use_graph else ()
        self.stem = Stem(cfg)
        self.film = (SubjectFiLM(n_subjects, cfg.subj_emb_dim, d)
                     if cfg.use_subject_film else None)
        self.blocks = nn.ModuleList([STBlock(cfg, N, 2 ** i, views)
                                     for i in range(cfg.n_st_blocks)])
        self.use_syn = cfg.use_synergy
        if self.use_syn:
            self.pool = SynergyPool(d, K)
            cviews = tuple(v for v in ("dyn", "learn") if v in views)
            self.coarse = STBlock(cfg, K, 1, cviews)
            self.ro_coarse = AttnReadout(d)
        self.ro_fine = AttnReadout(d)
        fin = 4 * d if self.use_syn else 2 * d
        self.head = nn.Sequential(nn.Linear(fin, cfg.head_hidden), nn.GELU(),
                                  nn.Dropout(cfg.p_drop_head))
        self.classifier = nn.Linear(cfg.head_hidden, cfg.n_classes)

    def forward(self, x, A_anat, A_coh, subj=None):
        h = self.stem(x)
        if self.film is not None:
            assert subj is not None, "use_subject_film=True requires subj_idx"
            h = self.film(h, subj)
        A_f = A_d = None
        for blk in self.blocks:
            h, A_f, A_d = blk(h, A_anat, A_coh)
        aux = dict(S=None, A_fused=A_f, A_dyn=A_d)
        parts = [self.ro_fine(h)]
        if self.use_syn:
            hc, S = self.pool(h)
            hc, _, _ = self.coarse(hc)
            parts.append(self.ro_coarse(hc))
            aux["S"] = S
        z = self.head(torch.cat(parts, dim=-1))
        return self.classifier(z), z, aux


def aux_losses(aux, device=DEVICE):
    """Tolerant of ablations that remove the synergy pool or the graph entirely."""
    zero = torch.zeros((), device=device)
    S = aux.get("S"); A = aux.get("A_fused"); Ad = aux.get("A_dyn")
    link = ((A.float() - S.float() @ S.float().transpose(1, 2)) ** 2).mean() \
        if (S is not None and A is not None) else zero
    ent = -(S.float().clamp_min(1e-9).log() * S.float()).sum(-1).mean() \
        if S is not None else zero
    l1 = Ad.float().abs().mean() if Ad is not None else zero
    return link, ent, l1


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


print(f"{'variant':<34}{'params':>10}  forward check")
for _lbl, _ov in [("A0 full", {}),
                  ("A1 no graph", dict(use_graph=False)),
                  ("A1b uniform only", dict(graph_views=("uniform",))),
                  ("A4 A_learn only", dict(graph_views=("learn",))),
                  ("A7 no synergy pool", dict(use_synergy=False)),
                  ("P1 no FiLM", dict(use_subject_film=False)),
                  ("A15b d_model=96", dict(d_model=96))]:
    _c = dc_replace(BASE, **_ov)
    _m = MVSTGNN(_c, 20).to(DEVICE)
    _x = torch.randn(4, _c.n_channels, _c.win, device=DEVICE)
    _ac = A_ANAT.unsqueeze(0).expand(4, 12, 12).contiguous()
    _s = torch.zeros(4, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        _lg, _z, _aux = _m(_x, A_ANAT, _ac, _s if _c.use_subject_film else None)
        _lk, _en, _l1 = aux_losses(_aux)
    assert _lg.shape == (4, _c.n_classes)
    assert all(torch.isfinite(t) for t in (_lk, _en, _l1))
    print(f"{_lbl:<34}{n_params(_m):>10,}  logits {tuple(_lg.shape)}  aux finite")
    del _m
torch.cuda.empty_cache()
print("\n[PASS] every ablation variant builds, runs, and has finite aux losses")

variant                               params  forward check
A0 full                              186,059  logits (4, 5)  aux finite
A1 no graph                          151,499  logits (4, 5)  aux finite
A1b uniform only                     163,979  logits (4, 5)  aux finite
A4 A_learn only                      169,419  logits (4, 5)  aux finite
A7 no synergy pool                   139,334  logits (4, 5)  aux finite
P1 no FiLM                           183,563  logits (4, 5)  aux finite
A15b d_model=96                      332,299  logits (4, 5)  aux finite

[PASS] every ablation variant builds, runs, and has finite aux losses


---
## Step 5 — Ablation registry and cost

Tiers are ordered by how much the answer would change what you do next.

**Tier 1 — the make-or-break set.** If `A1 ≈ A0`, the graph is not the contribution. If
`A9`/`P1 ≈ A0`, the gain is the shared toolkit every baseline already has.

**Tier 2** — which views and components carry the load.
**Tier 3** — sensitivity sweeps (synergy count, capacity, window length).



| Mode | Variants | Cost |
|---|---|---|
| Tier 1, all 5 folds | 4 | **~7.5 h** |
| Tier 1+2, all 5 folds | 13 | ~24 h |
| **Screen everything on fold 0** (`FOLDS_TO_RUN=(0,)`) | 25 | ~9 h |
| Everything, all 5 folds | 25 | ~47 h |



In [ ]:
# (id, label, tier, overrides)
ABLATIONS = [
    ("A0",   "full MV-STGNN (reference)",          1, {}),
    ("A1",   "no graph (independent nodes)",       1, dict(use_graph=False)),
    ("A9",   "no SupCon auxiliary",                1, dict(w_supcon=0.0)),
    ("P1",   "no subject FiLM",                    1, dict(use_subject_film=False)),

    ("A1b",  "uniform adjacency only",             2, dict(graph_views=("uniform",))),
    ("A2",   "A_anat only",                        2, dict(graph_views=("anat",))),
    ("A3",   "A_dyn only",                         2, dict(graph_views=("dyn",))),
    ("A4",   "A_learn only",                       2, dict(graph_views=("learn",))),
    ("A5",   "no A_anat (drop the prior)",         2, dict(graph_views=("coh", "dyn", "learn"))),
    ("A6",   "no A_coh (drop train-fit view)",     2, dict(graph_views=("anat", "dyn", "learn"))),
    ("A7",   "no synergy pooling",                 2, dict(use_synergy=False)),
    ("A10",  "no channel dropout",                 2, dict(aug_chan_drop_p=0.0)),
    ("A11",  "dense A_dyn (no top-k)",             2, dict(graph_topk=12)),
    ("P2",   "global A_coh (population mean)",     2, dict(coh_global=True)),
    ("P3",   "BatchNorm instead of GroupNorm",     2, dict(stem_norm="batch")),

    ("A8a",  "K=2 synergies",                      3, dict(n_synergies=2)),
    ("A8b",  "K=6 synergies",                      3, dict(n_synergies=6)),
    ("A8c",  "K=8 synergies",                      3, dict(n_synergies=8)),
    ("A15a", "d_model=32",                         3, dict(d_model=32)),
    ("A15b", "d_model=96",                         3, dict(d_model=96)),
    ("A12a", "window 100 ms",                      3, dict(win_ms=100)),
    ("A12b", "window 150 ms",                      3, dict(win_ms=150)),
    ("A12c", "window 300 ms",                      3, dict(win_ms=300)),
    ("A12d", "window 400 ms",                      3, dict(win_ms=400)),
    ("A13",  "train stride 10 ms (4x data)",       3, dict(train_stride_ms=10)),
]

DATA_CHANGING = {"win_ms", "train_stride_ms", "eval_stride_ms", "trim_ms"}

print(f"{'id':<7}{'tier':>5}  {'params':>9}  {'data?':>6}  label")
_seen = set()
for aid, lbl, tier, ov in ABLATIONS:
    assert aid not in _seen, f"duplicate ablation id {aid}"
    _seen.add(aid)
    c = dc_replace(BASE, **ov)
    dch = bool(set(ov) & DATA_CHANGING)
    m = MVSTGNN(c, len(SUBJECTS))
    print(f"{aid:<7}{tier:>5}  {n_params(m):>9,}  {'YES' if dch else '-':>6}  {lbl}")
    del m
print(f"\n{len(ABLATIONS)} variants | tier1 {sum(1 for a in ABLATIONS if a[2]==1)}"
      f" tier2 {sum(1 for a in ABLATIONS if a[2]==2)}"
      f" tier3 {sum(1 for a in ABLATIONS if a[2]==3)}")
print("data?=YES -> changes the windows, so it gets its own split fingerprint and is a")
print("            SWEEP rather than a same-window paired ablation (still subject-paired)")

id      tier     params   data?  label
A0         1    186,059       -  full MV-STGNN (reference)
A1         1    151,499       -  no graph (independent nodes)
A9         1    186,059       -  no SupCon auxiliary
P1         1    183,563       -  no subject FiLM
A1b        2    163,979       -  uniform adjacency only
A2         2    163,979       -  A_anat only
A3         2    184,779       -  A_dyn only
A4         2    169,419       -  A_learn only
A5         2    184,511       -  no A_anat (drop the prior)
A6         2    184,511       -  no A_coh (drop train-fit view)
A7         2    139,334       -  no synergy pooling
A10        2    186,059       -  no channel dropout
A11        2    186,059       -  dense A_dyn (no top-k)
P2         2    186,059       -  global A_coh (population mean)
P3         2    186,059       -  BatchNorm instead of GroupNorm
A8a        3    185,865       -  K=2 synergies
A8b        3    186,253       -  K=6 synergies
A8c        3    186,447       -  K=8 syne

---
## Step 6 — Training (identical loop to notebook 02)

Same optimiser, cosine schedule with warm-up, AMP, clipping, class weights, label smoothing,
early stopping on val rep 3, best-weight restore, and a single touch of the test set. Only the
config differs between variants.

In [ ]:
def evaluate(model, src, win, A_anat, A_coh_all, cfg):
    model.eval()
    b = Batcher(win)
    P = []
    amp_on = cfg.amp and BF16 and DEVICE.type == "cuda"
    with torch.no_grad():
        for st, _, sb in b.epoch(cfg.eval_batch, shuffle=False):
            x = src.gather(st, train=False)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp_on):
                logits, _, _ = model(x, A_anat, A_coh_all[sb],
                                     sb if cfg.use_subject_film else None)
            P.append(logits.float().softmax(-1).cpu().numpy())
    prob = np.concatenate(P) if P else np.zeros((0, cfg.n_classes), np.float32)
    return prob.argmax(1), prob


def train_variant_fold(cfg, fold, seed=0, verbose=True):
    set_seed(seed)
    t_setup = time.time()
    test_rep = cfg.test_reps[fold]
    train_reps = cfg.train_reps(test_rep)

    src = PooledSource(N_TOTAL, cfg)
    nfl = 0
    A_list = []
    for si, s in enumerate(SUBJECTS):
        scale, f = compute_scale(SUBJ[s]["sig"], SUBJ[s]["segs"], train_reps, cfg)
        src.write(OFFSETS[si], SUBJ[s]["sig"], scale)
        nfl += f
        A_list.append(compute_A_coh(SUBJ[s]["sig"], SUBJ[s]["segs"], train_reps, cfg))
    A_coh_all = torch.from_numpy(np.stack(A_list)).to(DEVICE)
    if cfg.coh_global:                      # P2: replace per-subject with population mean
        A_coh_all = A_coh_all.mean(0, keepdim=True).expand(len(SUBJECTS), -1, -1).contiguous()

    tr, va, te, meta = fold_indices(SUBJ, SUBJECTS, OFFSETS, fold, cfg)
    if cfg.cap_train_per_class:
        tr = cap_train_windows(tr, cfg.n_classes, seed)
        meta["n_train"] = len(tr["start"])

    model = MVSTGNN(cfg, len(SUBJECTS)).to(DEVICE)
    w = make_class_weights(tr["label"], cfg.n_classes) if cfg.class_weighted else None
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    spe = max(1, math.ceil(len(tr["start"]) / cfg.batch_size))
    total, warm = cfg.epochs * spe, cfg.warmup_epochs * spe

    def lr_at(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    rng = np.random.default_rng(seed)
    amp_on = cfg.amp and BF16 and DEVICE.type == "cuda"
    btr = Batcher(tr)
    if verbose:
        print(f"      setup {time.time()-t_setup:.0f}s | params {n_params(model):,} | "
              f"train {meta['n_train']:,} | {spe} steps/ep | floored {nfl}")

    best_f1, best_state, best_ep, bad, hist = -1.0, None, -1, 0, []
    t0 = time.time()
    for ep in range(cfg.epochs):
        model.train()
        tot = seen = 0
        for st, yb, sb in btr.epoch(cfg.batch_size, True, rng):
            x = src.gather(st, train=True, gen=gen)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp_on):
                logits, z, aux = model(x, A_ANAT, A_coh_all[sb],
                                       sb if cfg.use_subject_film else None)
                loss = F.cross_entropy(logits, yb, weight=w,
                                       label_smoothing=cfg.label_smoothing)
                if cfg.w_supcon > 0:
                    loss = loss + cfg.w_supcon * supcon_loss(z, yb, cfg.supcon_tau)
                lk, en, l1 = aux_losses(aux)
                loss = loss + cfg.w_linkpred * lk + cfg.w_entropy * en + cfg.w_adj_l1 * l1
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at ep {ep}")
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sched.step()
            tot += loss.item() * len(yb); seen += len(yb)

        pv, _ = evaluate(model, src, va, A_ANAT, A_coh_all, cfg)
        vf1 = f1_score(va["label"], pv, average="macro", zero_division=0)
        hist.append(dict(epoch=ep, train_loss=tot / max(1, seen), val_macro_f1=vf1))
        if vf1 > best_f1:
            best_f1, best_ep, bad = vf1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose and (ep % 20 == 0 or ep == cfg.epochs - 1):
            print(f"        ep {ep:>2}  loss {tot/max(1,seen):.4f}  val_f1 {vf1:.4f}  "
                  f"best@{best_ep}  {(time.time()-t0)/60:.1f}m")
        if bad >= cfg.patience:
            if verbose:
                print(f"        early stop ep {ep} (best {best_ep})")
            break

    model.load_state_dict(best_state)
    pt, prob = evaluate(model, src, te, A_ANAT, A_coh_all, cfg)
    common = dict(fold=fold, seed=seed, test_rep=test_rep, n_train=meta["n_train"],
                  best_epoch=best_ep, epochs_run=len(hist), val_macro_f1=best_f1,
                  n_params=n_params(model), train_min=(time.time() - t0) / 60.0)
    rows = per_subject_rows(te, pt, SUBJECTS, cfg.n_classes)
    for r in rows:
        r.update(common)
    pooled = metrics_from(te["label"], pt, te["seg_id"], cfg.n_classes)
    pooled.update(common)
    preds = dict(y_true=te["label"], y_pred=pt, y_prob=prob,
                 subj_idx=te["subj_idx"], seg_id=te["seg_id"], rep=te["rep"])
    del src, model, best_state, btr, A_coh_all
    torch.cuda.empty_cache()
    return rows, pooled, preds, hist

---
## Step 7 — Run the grid

Resumable per `(variant, fold)` and keyed by the variant's own config hash, so editing a
hyperparameter can never silently reuse a stale result.

In [ ]:
TIERS_TO_RUN = (1,)             # (1,) -> ~7.5 h | (1,2) -> ~24 h | (1,2,3) -> ~47 h
FOLDS_TO_RUN = None            # None = all 5. (0,) = fast screen on the hardest fold.
SEEDS = (0,)
VERBOSE = True

folds = list(range(BASE.n_folds)) if FOLDS_TO_RUN is None else list(FOLDS_TO_RUN)
todo = [a for a in ABLATIONS if a[2] in TIERS_TO_RUN]
est_h = len(todo) * len(folds) * len(SEEDS) * 22 / 60
print(f"{len(todo)} variants x {len(folds)} folds x {len(SEEDS)} seed(s) "
      f"= {len(todo)*len(folds)*len(SEEDS)} runs  ~{est_h:.1f} h")
for a in todo:
    print(f"   {a[0]:<7} {a[1]}")


def run_ablations(todo, folds, seeds=(0,), resume=True, verbose=True):
    sub_all, pol_all = [], []
    t_start = time.time()
    for aid, lbl, tier, ov in todo:
        cfg = dc_replace(BASE, **ov)
        h = cfg_hash(cfg)
        dch = bool(set(ov) & DATA_CHANGING)
        fp = split_fingerprint(SUBJ, SUBJECTS, OFFSETS, cfg) if dch else BASE_FP
        ck = RUNS / f"{aid}_{h}"
        ck.mkdir(parents=True, exist_ok=True)
        print(f"\n{'='*72}\n{aid}  {lbl}\n  hash {h} | fingerprint {fp}"
              f"{'  (DATA CHANGED)' if dch else ''}\n{'='*72}")
        for seed in seeds:
            for fold in folds:
                fs_ = ck / f"f{fold}_s{seed}_per_subject.csv"
                fp_ = ck / f"f{fold}_s{seed}_pooled.csv"
                npz = PREDS / f"{aid}_{h}_f{fold}_s{seed}.npz"
                if resume and fs_.exists() and fp_.exists() and npz.exists():
                    d1, d2 = pd.read_csv(fs_), pd.read_csv(fp_)
                    sub_all.append(d1); pol_all.append(d2)
                    print(f"  [fold {fold}] cached  bal_acc {d2['bal_acc'].iloc[0]:.4f}")
                    continue
                print(f"  [fold {fold}] test rep {cfg.test_reps[fold]}")
                rows, pooled, preds, hist = train_variant_fold(cfg, fold, seed, verbose)
                for r in rows:
                    r.update(ablation=aid, label=lbl, tier=tier,
                             split_fingerprint=fp, data_changed=dch)
                pooled.update(ablation=aid, label=lbl, tier=tier,
                              split_fingerprint=fp, data_changed=dch)
                d1, d2 = pd.DataFrame(rows), pd.DataFrame([pooled])
                # re-assert the dirs: a long run can outlive an external cleanup
                fs_.parent.mkdir(parents=True, exist_ok=True)
                npz.parent.mkdir(parents=True, exist_ok=True)
                d1.to_csv(fs_, index=False); d2.to_csv(fp_, index=False)
                np.savez_compressed(npz, ablation=aid, cfg_hash=h, split_fingerprint=fp,
                                    fold=fold, seed=seed,
                                    subjects=np.array(SUBJECTS), **preds)
                sub_all.append(d1); pol_all.append(d2)
                print(f"    -> bal_acc {pooled['bal_acc']:.4f}  "
                      f"macro_f1 {pooled['macro_f1']:.4f}  vote {pooled['vote_acc']:.4f}  "
                      f"({pooled['train_min']:.1f} min, elapsed "
                      f"{(time.time()-t_start)/60:.0f} min)")
    return (pd.concat(sub_all, ignore_index=True) if sub_all else pd.DataFrame(),
            pd.concat(pol_all, ignore_index=True) if pol_all else pd.DataFrame())


df_sub, df_pol = run_ablations(todo, folds, SEEDS, resume=True, verbose=VERBOSE)
print(f"\ndone: {len(df_sub)} per-subject rows, {len(df_pol)} pooled rows")

4 variants x 5 folds x 1 seed(s) = 20 runs  ~7.3 h
   A0      full MV-STGNN (reference)
   A1      no graph (independent nodes)
   A9      no SupCon auxiliary
   P1      no subject FiLM

A0  full MV-STGNN (reference)
  hash d2429ed3 | fingerprint 3fed90994b841e0d
  [fold 0] test rep 1
      setup 6s | params 186,059 | train 46,889 | 184 steps/ep | floored 2
        ep  0  loss 2.2670  val_f1 0.2530  best@0  0.3m
        ep 20  loss 1.0096  val_f1 0.8090  best@19  5.4m
        ep 40  loss 0.8626  val_f1 0.8428  best@39  10.3m
        early stop ep 51 (best 39)
    -> bal_acc 0.7164  macro_f1 0.7170  vote 0.9100  (13.1 min, elapsed 13 min)
  [fold 1] test rep 2
      setup 5s | params 186,059 | train 47,173 | 185 steps/ep | floored 4
        ep  0  loss 2.2684  val_f1 0.2173  best@0  0.3m
        ep 20  loss 1.0208  val_f1 0.7988  best@20  5.2m
        ep 40  loss 0.8760  val_f1 0.8175  best@31  10.1m
        early stop ep 43 (best 31)
    -> bal_acc 0.7912  macro_f1 0.7939  vote 0.9600 

---
## Step 8 — Ablation table

`Δ` is the **paired per-subject** difference `A0 − variant` in balanced accuracy: positive means
removing that component *hurt*, i.e. the component was doing work. `win/loss` counts subjects.

Read `Δ` together with `best_ep/run`: a variant that stopped early because it converged is a fair
comparison, one that ran to the cap may simply be under-trained.

In [ ]:
METRICS = ["acc", "bal_acc", "macro_f1", "weighted_f1", "kappa", "vote_acc"]

per_subject = (df_sub.groupby(["ablation", "subject", "fold"], as_index=False)[METRICS].mean()
               .groupby(["ablation", "subject"], as_index=False)[METRICS].mean())
info = df_sub.groupby("ablation").agg(
    label=("label", "first"), tier=("tier", "first"),
    data_changed=("data_changed", "first"), n_params=("n_params", "first"),
    best_epoch=("best_epoch", "mean"), epochs_run=("epochs_run", "mean"),
    train_min=("train_min", "mean")).reset_index()

assert "A0" in set(per_subject["ablation"]), "A0 not run — nothing to compare against"
ref = per_subject[per_subject.ablation == "A0"].set_index("subject")["bal_acc"]

rows = []
for aid in per_subject["ablation"].unique():
    d = per_subject[per_subject.ablation == aid].set_index("subject")["bal_acc"]
    com = ref.index.intersection(d.index)
    diff = (ref.loc[com] - d.loc[com]).values
    i = info[info.ablation == aid].iloc[0]
    rows.append(dict(
        ablation=aid, tier=int(i["tier"]), label=i["label"],
        params=int(i["n_params"]), data=bool(i["data_changed"]),
        bal_acc=d.mean(), sd=d.std(ddof=1),
        delta=diff.mean() if aid != "A0" else 0.0,
        d_sd=diff.std(ddof=1) if aid != "A0" else 0.0,
        win=int((diff > 0).sum()) if aid != "A0" else 0,
        loss=int((diff < 0).sum()) if aid != "A0" else 0,
        best_ep=i["best_epoch"], run=i["epochs_run"], min=i["train_min"]))
tab = pd.DataFrame(rows).sort_values(["tier", "delta"], ascending=[True, False])

print("=" * 112)
print(f"ABLATION TABLE — {BASE.n_classes}-class, {len(BASE.subjects)} subjects, "
      f"folds {folds}, chance {1.0/BASE.n_classes:.4f}")
print("=" * 112)
print(f"{'id':<7}{'T':>2} {'bal_acc':>8}{'sd':>7}{'delta':>9}{'+/-':>7}"
      f"{'win':>5}{'los':>5}{'params':>9}{'best/run':>10}{'min':>6}  label")
for _, r in tab.iterrows():
    dl = "  ref  " if r["ablation"] == "A0" else f"{r['delta']:+.4f}"
    ds = "     -" if r["ablation"] == "A0" else f"{r['d_sd']:.4f}"
    print(f"{r['ablation']:<7}{r['tier']:>2} {r['bal_acc']:>8.4f}{r['sd']:>7.4f}"
          f"{dl:>9}{ds:>7}{r['win']:>5}{r['loss']:>5}{r['params']:>9,}"
          f"{r['best_ep']:>6.0f}/{r['run']:<3.0f}{r['min']:>6.1f}  "
          f"{r['label']}{'  [data changed]' if r['data'] else ''}")

print("\ndelta = A0 - variant, paired per subject. POSITIVE = the component helps.")

# ---- the decisive read-out ------------------------------------------------
print("\n" + "=" * 112)
print("VERDICT ON THE HEADLINE QUESTIONS")
print("=" * 112)
for aid, q in (("A1", "does the graph do anything?"),
               ("A9", "is the gain just SupCon?"),
               ("P1", "is the gain just subject FiLM?")):
    if aid not in set(tab["ablation"]):
        print(f"  {aid:<5} NOT RUN — {q}")
        continue
    r = tab[tab.ablation == aid].iloc[0]
    n = r["win"] + r["loss"]
    if r["delta"] < 0.005:
        v = "NO EFFECT -> this component is not the contribution. Report it."
    elif r["delta"] < 0.02:
        v = "small effect -> needs the significance test before claiming it"
    else:
        v = "clear effect"
    print(f"  {aid:<5} delta {r['delta']:+.4f} (sd {r['d_sd']:.4f}, "
          f"{r['win']}/{n} subjects)  {q}\n        {v}")

ABLATION TABLE — 5-class, 20 subjects, folds [0, 1, 2, 3, 4], chance 0.2000
id      T  bal_acc     sd    delta    +/-  win  los   params  best/run   min  label
A1      1   0.7447 0.0952  +0.0734 0.0336   20    0  151,499    73/84   21.0  no graph (independent nodes)
P1      1   0.8057 0.0791  +0.0125 0.0148   17    3  183,563    49/62   15.1  no subject FiLM
A0      1   0.8182 0.0777    ref        -    0    0  186,059    37/50   12.3  full MV-STGNN (reference)
A9      1   0.8274 0.0796  -0.0092 0.0171    7   13  186,059    60/73   18.2  no SupCon auxiliary

delta = A0 - variant, paired per subject. POSITIVE = the component helps.

VERDICT ON THE HEADLINE QUESTIONS
  A1    delta +0.0734 (sd 0.0336, 20/20 subjects)  does the graph do anything?
        clear effect
  A9    delta -0.0092 (sd 0.0171, 7/20 subjects)  is the gain just SupCon?
        NO EFFECT -> this component is not the contribution. Report it.
  P1    delta +0.0125 (sd 0.0148, 17/20 subjects)  is the gain just subject FiLM

---
## Step 9 — Artifacts

Same schema as the baselines notebook, so `05_significance.ipynb` can run comparison family
**F2** (A0 vs each ablation, Holm-corrected within the family, separately from the F1 baseline
family) without any reshaping.

In [ ]:
STAMP = time.strftime("%Y%m%d_%H%M%S")
TAG = f"ablations_{BASE.n_classes}cls_{BASE.win_ms}ms"


def save2(df, base):
    RESULTS.mkdir(parents=True, exist_ok=True)
    a = RESULTS / f"{TAG}_{base}_{STAMP}.csv"
    b = RESULTS / f"{TAG}_{base}_latest.csv"
    df.to_csv(a, index=False); df.to_csv(b, index=False)
    return [a, b]


ps = per_subject.merge(info[["ablation", "label", "tier", "data_changed", "n_params"]],
                       on="ablation", how="left")
ps["n_classes"] = BASE.n_classes
ps["protocol"] = "pooled_subject_mixed_repetition_split"

paths = []
paths += save2(ps, "long_per_subject")
paths += save2(df_sub, "long_per_fold_subject")
paths += save2(df_pol, "long_pooled_per_fold")
paths += save2(tab, "summary_table")

manifest = dict(
    stamp=STAMP, tag=TAG, kind="ablation",
    base_split_fingerprint=BASE_FP,
    base_cfg_hash=cfg_hash(BASE),
    reference_variant="A0",
    folds_run=folds, tiers_run=list(TIERS_TO_RUN), seeds=list(SEEDS),
    subjects=list(BASE.subjects), n_classes=BASE.n_classes,
    class_subset=list(BASE.class_subset) if BASE.class_subset else None,
    chance=1.0 / BASE.n_classes, epochs=BASE.epochs, patience=BASE.patience,
    variants={a[0]: dict(label=a[1], tier=a[2],
                         overrides=canon(a[3]),
                         cfg_hash=cfg_hash(dc_replace(BASE, **a[3])),
                         data_changed=bool(set(a[3]) & DATA_CHANGING))
              for a in todo},
    aggregation="seeds -> folds -> one value per subject; unit of analysis = subject",
    primary_endpoint="bal_acc",
    planned_tests=dict(
        family="F2 (ablations), corrected SEPARATELY from F1 (baselines)",
        posthoc="Wilcoxon signed-rank, A0 vs each ablation, Holm-corrected within F2",
        effect_size="Cliff's delta + Cohen's d_z",
        ci="BCa bootstrap over subjects, 10000 resamples"),
    caveats=[
        "per-subject scores come from ONE pooled model per fold -> not independent replications",
        "variants marked data_changed use different windows; they are subject-paired sweeps, "
        "not same-window ablations",
        "compare best_epoch against epochs_run: a variant at the cap may be under-trained "
        "rather than genuinely worse",
    ],
    pred_files=sorted(p.name for p in PREDS.glob("*.npz")),
)
for nm in (f"{TAG}_manifest_{STAMP}.json", f"{TAG}_manifest_latest.json"):
    (RESULTS / nm).write_text(json.dumps(manifest, indent=2))
    paths.append(RESULTS / nm)

print("wrote:")
for p in paths:
    print(f"  {_rel(p)}  ({p.stat().st_size/1024:.1f} KB)")
npzs = sorted(PREDS.glob("*.npz"))
print(f"\nprediction files: {len(npzs)} "
      f"({sum(p.stat().st_size for p in npzs)/1e6:.1f} MB)")

piv = ps.pivot_table(index="subject", columns="ablation", values="bal_acc")
print(f"\npaired matrix: {piv.shape[0]} subjects x {piv.shape[1]} variants  "
      f"missing {int(piv.isna().sum().sum())}")
assert piv.isna().sum().sum() == 0, "holes in the paired matrix -> Wilcoxon would drop subjects"
print("[PASS] complete rectangular paired matrix -> ready for family F2")
display(piv.round(4))

wrote:
  results\tables\ablations_5cls_200ms_long_per_subject_20260807_173750.csv  (14.7 KB)
  results\tables\ablations_5cls_200ms_long_per_subject_latest.csv  (14.7 KB)
  results\tables\ablations_5cls_200ms_long_per_fold_subject_20260807_173750.csv  (85.6 KB)
  results\tables\ablations_5cls_200ms_long_per_fold_subject_latest.csv  (85.6 KB)
  results\tables\ablations_5cls_200ms_long_pooled_per_fold_20260807_173750.csv  (4.4 KB)
  results\tables\ablations_5cls_200ms_long_pooled_per_fold_latest.csv  (4.4 KB)
  results\tables\ablations_5cls_200ms_summary_table_20260807_173750.csv  (0.7 KB)
  results\tables\ablations_5cls_200ms_summary_table_latest.csv  (0.7 KB)
  results\tables\ablations_5cls_200ms_manifest_20260807_173750.json  (2.8 KB)
  results\tables\ablations_5cls_200ms_manifest_latest.json  (2.8 KB)

prediction files: 20 (3.6 MB)

paired matrix: 20 subjects x 4 variants  missing 0
[PASS] complete rectangular paired matrix -> ready for family F2


ablation,A0,A1,A9,P1
subject,,,,
1,0.8830,0.8592,0.9086,0.8979
2,0.7501,0.7060,0.7727,0.7454
3,0.8810,0.8038,0.8822,0.8563
4,0.7158,0.5940,0.7302,0.7062
5,0.8913,0.7472,0.9065,0.8970
6,0.8832,0.8363,0.8978,0.8544
7,0.6488,0.5373,0.6653,0.6487
8,0.9378,0.8686,0.9498,0.9326
9,0.8975,0.8741,0.9258,0.8964
